# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.

In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [ ]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.

In [ ]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)

## Problem 1 - Part (a)
### Base Model Training and Evaluation

In [ ]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

X = df[feature_names].values
y = df["Class"].values

print("Feature matrix shape:", X.shape)
print("Label vector shape:  ", y.shape)

In [ ]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)

In [ ]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat  = to_categorical(y_test,  num_classes=num_classes)

print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat  shape:", y_test_cat.shape)

In [ ]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.summary()

In [ ]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_scaled, y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

train_acc = history.history['accuracy'][-1]
print(f"\nFinal Training Accuracy: {train_acc:.4f}")

In [ ]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

loss, test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss:     {loss:.4f}")

y_pred_probs = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test_cat,   axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Class 0', 'Class 1', 'Class 2']))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

In [ ]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

def file_size_kb(filename):
    """Return the size of a file in kilobytes."""
    return os.path.getsize(filename) / 1024

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_base = converter.convert()

with open('model_base.tflite', 'wb') as f:
    f.write(tflite_base)

print(f"Base (float32) TFLite model size: {file_size_kb('model_base.tflite'):.2f} KB")

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)

In [ ]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        # (b) Provide representative_data_gen(X_train_scaled).
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        # (d) Set inference_input_type and inference_output_type to tf.int8.
        converter.inference_input_type  = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        # (b) Set supported_types to [tf.float16].
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.
    tflite_model = converter.convert()
    with open(filename, 'wb') as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    in_scale,  in_zp  = input_details[0]['quantization']
    out_scale, out_zp = output_details[0]['quantization']

    y_true = np.argmax(y_test_cat, axis=1)
    y_pred = []

    for i in range(len(X_test)):
        sample = X_test[i:i + 1].astype(np.float32)

        # Quantize input if needed
        if input_details[0]['dtype'] == np.int8:
            if in_scale != 0:
                sample = np.round(sample / in_scale + in_zp).astype(np.int8)
            else:
                sample = sample.astype(np.int8)

        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]['index']).copy()

        # Dequantize output if needed
        if output_details[0]['dtype'] == np.int8:
            if out_scale != 0:
                output = (output.astype(np.float32) - out_zp) * out_scale

        y_pred.append(np.argmax(output))

    y_pred = np.array(y_pred)

    # Step 4: Report results.
    print(f'\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB')

    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=['Class 0', 'Class 1', 'Class 2']))
    print('Confusion Matrix:')
    print(confusion_matrix(y_true, y_pred))

In [ ]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'int8',    'model_int8.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'float16', 'model_float16.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'dynamic', 'model_dynamic.tflite')

## Problem 1 - Part (c)

### Pruning

In [ ]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

batch_size_prune = 8
epochs_prune     = 10
dataset_size     = len(X_train_scaled)
end_step         = math.ceil(dataset_size / batch_size_prune) * epochs_prune

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

print(f"end_step: {end_step}")

In [ ]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = Sequential([
    prune_low_magnitude(
        Dense(64, activation='relu', input_shape=(num_features,)),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        Dense(num_classes, activation='softmax'),
        pruning_schedule=pruning_schedule
    )
])

pruned_model.summary()

In [ ]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

pruned_history = pruned_model.fit(
    X_train_scaled, y_train_cat,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
tflite_pruned = converter.convert()

with open('model_pruned.tflite', 'wb') as f:
    f.write(tflite_pruned)

print(f"Pruned TFLite model size: {file_size_kb('model_pruned.tflite'):.2f} KB")

In [ ]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

y_pred_probs_pruned = stripped_model.predict(X_test_scaled)
y_pred_pruned = np.argmax(y_pred_probs_pruned, axis=1)
y_true        = np.argmax(y_test_cat, axis=1)

print("Classification Report (Pruned Model):")
print(classification_report(y_true, y_pred_pruned, target_names=['Class 0', 'Class 1', 'Class 2']))

print("Confusion Matrix (Pruned Model):")
print(confusion_matrix(y_true, y_pred_pruned))

## Problem 1 - Part (d)

### Knowledge Distillation

In [ ]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

student_model = Sequential([
    Dense(32, activation='relu', input_shape=(num_features,)),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

student_model.summary()

In [ ]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

teacher_preds_soft = model.predict(X_train_scaled)
print("Teacher soft label shape:", teacher_preds_soft.shape)

In [ ]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# (a) Concatenate hard and soft labels
y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1)
print("Combined label shape:", y_train_combined.shape)  # expected: (n_train, 6)


def distillation_loss(y_true_combined, y_pred):

    alpha = 0.5  # weight for hard vs. soft loss

    # Split the combined label into hard (ground truth) and soft (teacher) targets
    y_true_hard = y_true_combined[:, :num_classes]   # [:, :3]
    y_true_soft = y_true_combined[:, num_classes:]   # [:, 3:]

    # Compute hard-label loss
    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)

    # Compute soft-label loss
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    # Combine with alpha weighting
    return alpha * hard_loss + (1.0 - alpha) * soft_loss

In [ ]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

student_history = student_model.fit(
    X_train_scaled, y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd = converter.convert()

with open('model_kd.tflite', 'wb') as f:
    f.write(tflite_kd)

print(f"KD Student TFLite model size: {file_size_kb('model_kd.tflite'):.2f} KB")

In [ ]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

y_pred_probs_kd = student_model.predict(X_test_scaled)
y_pred_kd = np.argmax(y_pred_probs_kd, axis=1)
y_true    = np.argmax(y_test_cat, axis=1)

print("Classification Report (KD Student Model):")
print(classification_report(y_true, y_pred_kd, target_names=['Class 0', 'Class 1', 'Class 2']))

print("Confusion Matrix (KD Student Model):")
print(confusion_matrix(y_true, y_pred_kd))

## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.

### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.

In [ ]:
# Part (e): Further size reduction — KD student model + INT8 quantization
#
# Strategy: apply full INT8 quantization to the already-smaller student model.
# The student (32-16-3) is smaller than the teacher (64-32-3), and INT8 further
# reduces each weight from 32-bit float to 8-bit integer (~4x compression).

converter_e = tf.lite.TFLiteConverter.from_keras_model(student_model)
converter_e.optimizations = [tf.lite.Optimize.DEFAULT]
converter_e.representative_dataset = lambda: representative_data_gen(X_train_scaled)
converter_e.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_e.inference_input_type  = tf.int8
converter_e.inference_output_type = tf.int8

tflite_kd_int8 = converter_e.convert()

with open('model_kd_int8.tflite', 'wb') as f:
    f.write(tflite_kd_int8)

print(f"KD Student + INT8 TFLite model size: {file_size_kb('model_kd_int8.tflite'):.2f} KB")

# Evaluate with TFLite interpreter
interpreter_e = tf.lite.Interpreter(model_path='model_kd_int8.tflite')
interpreter_e.allocate_tensors()

in_det_e  = interpreter_e.get_input_details()
out_det_e = interpreter_e.get_output_details()

in_scale_e,  in_zp_e  = in_det_e[0]['quantization']
out_scale_e, out_zp_e = out_det_e[0]['quantization']

y_true_e = np.argmax(y_test_cat, axis=1)
y_pred_e = []

for i in range(len(X_test_scaled)):
    sample = X_test_scaled[i:i + 1].astype(np.float32)
    if in_det_e[0]['dtype'] == np.int8:
        if in_scale_e != 0:
            sample = np.round(sample / in_scale_e + in_zp_e).astype(np.int8)
        else:
            sample = sample.astype(np.int8)
    interpreter_e.set_tensor(in_det_e[0]['index'], sample)
    interpreter_e.invoke()
    output = interpreter_e.get_tensor(out_det_e[0]['index']).copy()
    if out_det_e[0]['dtype'] == np.int8 and out_scale_e != 0:
        output = (output.astype(np.float32) - out_zp_e) * out_scale_e
    y_pred_e.append(np.argmax(output))

y_pred_e = np.array(y_pred_e)

print("\nClassification Report (KD Student + INT8):")
print(classification_report(y_true_e, y_pred_e, target_names=['Class 0', 'Class 1', 'Class 2']))
print("Confusion Matrix (KD Student + INT8):")
print(confusion_matrix(y_true_e, y_pred_e))

# Size comparison across all models
print("\n--- Model Size Comparison ---")
print(f"  Base float32:         {file_size_kb('model_base.tflite'):.2f} KB")
print(f"  Dynamic quantization: {file_size_kb('model_dynamic.tflite'):.2f} KB")
print(f"  Float16 quantization: {file_size_kb('model_float16.tflite'):.2f} KB")
print(f"  INT8 quantization:    {file_size_kb('model_int8.tflite'):.2f} KB")
print(f"  Pruned model:         {file_size_kb('model_pruned.tflite'):.2f} KB")
print(f"  KD student:           {file_size_kb('model_kd.tflite'):.2f} KB")
print(f"  KD student + INT8:    {file_size_kb('model_kd_int8.tflite'):.2f} KB")

# Problem 2: Exploring Edge Impulse (20 points)

### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.